# データ取得と方向予測の初期実験

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 1

出典: `FX.ipynb`、0始まりのindex=0。コード内容は変更していません。

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

df = yf.download(
    "JPY=X",
    start="2025-01-01",
    end="2026-09-01"
)

print(df.head())

df["Close"].plot(figsize=(12, 5))
plt.title("USD/JPY")
plt.show()

## 元Notebookのセル 2

出典: `FX.ipynb`、0始まりのindex=1。コード内容は変更していません。

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

df = yf.download(
    "JPY=X",
    start="2026-09-04",
    end="2026-09-05",
    interval="5m"
)

df["Close"].plot(figsize=(14, 6))

plt.title("USD/JPY 5-minute chart")
plt.xlabel("Time")
plt.ylabel("USD/JPY")
plt.grid()

plt.show()

## 元Notebookのセル 3

出典: `FX.ipynb`、0始まりのindex=2。コード内容は変更していません。

In [ ]:
%pip install mplfinance

## 元Notebookのセル 4

出典: `FX.ipynb`、0始まりのindex=3。コード内容は変更していません。

In [ ]:
import yfinance as yf
import mplfinance as mpf

# USD/JPYの5分足データを取得
df = yf.download(
    "JPY=X",
    start="2026-09-04",
    end="2026-09-05",
    interval="5m",
    auto_adjust=False
)

# yfinanceの列名をmplfinanceで扱える形にする
if df.columns.nlevels > 1:
    df.columns = df.columns.get_level_values(0)

# ローソク足チャートを表示
mpf.plot(
    df,
    type="candle",
    style="yahoo",
    title="USD/JPY 5-minute chart",
    ylabel="USD/JPY",
    figsize=(14, 7)
)

## 元Notebookのセル 5

出典: `FX.ipynb`、0始まりのindex=4。コード内容は変更していません。

In [ ]:
# =========================
# 1. ライブラリを読み込む
# =========================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


# =========================
# 2. USD/JPYの5分足データを取得
# =========================

df = yf.download(
    "JPY=X",
    period="60d",
    interval="5m",
    auto_adjust=False
)

# yfinanceの列が2段構造なら1段に直す
if df.columns.nlevels > 1:
    df.columns = df.columns.get_level_values(0)

print("取得したデータ数:", len(df))
print(df.head())


# =========================
# 3. 特徴量を作る
# =========================

# 5分間の変化率
df["return_5m"] = df["Close"].pct_change(1)

# 15分間の変化率
# 5分足 × 3本 = 15分
df["return_15m"] = df["Close"].pct_change(3)

# 30分間の変化率
# 5分足 × 6本 = 30分
df["return_30m"] = df["Close"].pct_change(6)


# 移動平均
df["MA5"] = df["Close"].rolling(5).mean()
df["MA20"] = df["Close"].rolling(20).mean()


# 現在価格が移動平均から何％離れているか
df["MA5_distance"] = df["Close"] / df["MA5"] - 1
df["MA20_distance"] = df["Close"] / df["MA20"] - 1


# 1本のローソク足の値幅
df["range"] = (
    df["High"] - df["Low"]
) / df["Close"]


# 過去1時間のボラティリティ
# 5分足 × 12本 = 60分
df["volatility_1h"] = (
    df["return_5m"]
    .rolling(12)
    .std()
)


# 時刻
df["hour"] = df.index.hour


# 曜日
# 月曜=0
# 火曜=1
# ...
# 金曜=4
df["weekday"] = df.index.dayofweek


# =========================
# 4. 30分後の値動きを正解データにする
# =========================

# 6本先 = 30分後
df["future_return_30m"] = (
    df["Close"].shift(-6)
    / df["Close"]
    - 1
)


# 30分後に上昇していたら1
# 下落または同値なら0
df["target"] = (
    df["future_return_30m"] > 0
).astype(int)


# =========================
# 5. AIに渡す特徴量を決める
# =========================

features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "MA5_distance",
    "MA20_distance",
    "range",
    "volatility_1h",
    "hour",
    "weekday"
]


# 必要な列だけ使う
# 欠損値は削除
data = df[
    features
    + ["target", "future_return_30m"]
].dropna()


print("\n機械学習に使えるデータ数:", len(data))


# =========================
# 6. 学習用とテスト用に時間順で分割
# =========================

# 古い80%を学習
# 新しい20%をテスト
split = int(len(data) * 0.8)

train = data.iloc[:split]
test = data.iloc[split:]


# 入力データ
X_train = train[features]
X_test = test[features]


# 正解データ
y_train = train["target"]
y_test = test["target"]


print("学習データ数:", len(train))
print("テストデータ数:", len(test))


# =========================
# 7. Random Forestを作る
# =========================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)


# =========================
# 8. AIに学習させる
# =========================

model.fit(
    X_train,
    y_train
)


# =========================
# 9. テストデータを予測
# =========================

# 0か1で予測
pred = model.predict(X_test)


# 上昇確率を取得
prob = model.predict_proba(X_test)[:, 1]


# =========================
# 10. AIの性能を評価
# =========================

accuracy = accuracy_score(
    y_test,
    pred
)

auc = roc_auc_score(
    y_test,
    prob
)


print("\n=========================")
print("モデル評価")
print("=========================")

print("Accuracy:", round(accuracy, 4))
print("ROC-AUC :", round(auc, 4))

print("\n詳細")
print(
    classification_report(
        y_test,
        pred
    )
)


# =========================
# 11. 最新の上昇確率を見る
# =========================

latest_X = data[features].iloc[[-1]]

latest_probability = model.predict_proba(
    latest_X
)[0, 1]


print("=========================")
print("最新の予測")
print("=========================")

print(
    "30分後の上昇確率:",
    round(
        latest_probability * 100,
        2
    ),
    "%"
)


# =========================
# 12. テスト期間の予測確率を保存
# =========================

results = test.copy()

results["prediction"] = pred
results["up_probability"] = prob


# =========================
# 13. 上昇確率をグラフ化
# =========================

plt.figure(
    figsize=(14, 6)
)

plt.plot(
    results.index,
    results["up_probability"]
)

plt.axhline(
    0.6,
    linestyle="--"
)

plt.axhline(
    0.4,
    linestyle="--"
)

plt.title(
    "USD/JPY 30-minute Up Probability"
)

plt.xlabel("Time")

plt.ylabel(
    "Probability"
)

plt.grid()

plt.show()


# =========================
# 14. 特徴量の重要度を見る
# =========================

importance = pd.Series(
    model.feature_importances_,
    index=features
)

importance = importance.sort_values(
    ascending=False
)


print("\n=========================")
print("特徴量の重要度")
print("=========================")

print(importance)


importance.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title(
    "Feature Importance"
)

plt.ylabel(
    "Importance"
)

plt.grid()

plt.show()